# NYC Yellow Taxi Trip Analysis — pandas
Each question below is answered in pandas and mirrors the matching query in **`trips.sql`**.
Every result was reconciled 1:1 against the SQL engine (DuckDB) on the same data — see the project's reconciliation note.

Source: NYC TLC Yellow Taxi trip records, January 2015 (sampled).

## Setup

In [1]:
import pandas as pdimport numpy as nppd.set_option('display.float_format', lambda v: f'{v:,.2f}')

Load the data and parse the pickup / dropoff timestamps once.

In [2]:
trips = pd.read_csv(r"C:\Users\amrel\Downloads\yellow_tripdata_2015-01__.csv")trips['tpep_pickup_datetime']  = pd.to_datetime(trips['tpep_pickup_datetime'])trips['tpep_dropoff_datetime'] = pd.to_datetime(trips['tpep_dropoff_datetime'])trips.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,2,2015-01-15 19:05:00,2015-01-15 19:23:00,1,1.59,-73.99,40.75,1,N,-73.97,40.75,1,12.00,1.00,0.50,3.25,0.00,0.30,17.05
1,1,2015-01-10 20:33:00,2015-01-10 20:53:00,1,3.30,-74.00,40.72,1,N,-73.99,40.76,1,14.50,0.50,0.50,2.00,0.00,0.30,17.80
2,1,2015-01-10 20:33:00,2015-01-10 20:43:00,1,1.80,-73.96,40.80,1,N,-73.95,40.82,2,9.50,0.50,0.50,0.00,0.00,0.30,10.80
3,1,2015-01-10 20:33:00,2015-01-10 20:35:00,1,0.50,-74.01,40.71,1,N,-74.00,40.72,2,3.50,0.50,0.50,0.00,0.00,0.30,4.80
4,1,2015-01-10 20:33:00,2015-01-10 20:52:00,1,3.00,-73.97,40.76,1,N,-74.00,40.74,2,15.00,0.50,0.50,0.00,0.00,0.30,16.30


Schema and shape overview.

In [3]:
trips.info()

<class 'pandas.DataFrame'>
RangeIndex: 32273 entries, 0 to 32272
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               32273 non-null  int64         
 1   tpep_pickup_datetime   32273 non-null  datetime64[us]
 2   tpep_dropoff_datetime  32273 non-null  datetime64[us]
 3   passenger_count        32273 non-null  int64         
 4   trip_distance          32273 non-null  float64       
 5   pickup_longitude       32273 non-null  float64       
 6   pickup_latitude        32273 non-null  float64       
 7   RateCodeID             32273 non-null  int64         
 8   store_and_fwd_flag     32273 non-null  str           
 9   dropoff_longitude      32273 non-null  float64       
 10  dropoff_latitude       32273 non-null  float64       
 11  payment_type           32273 non-null  int64         
 12  fare_amount            32273 non-null  float64       
 13  extra       

Summary statistics for the numeric columns.

In [4]:
trips.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,"32,273.00",1.52,1.00,1.00,2.00,2.00,2.00,0.50
tpep_pickup_datetime,32273,2015-01-16 14:05:31.720633,2015-01-01 01:08:00,2015-01-09 07:44:00,2015-01-15 22:43:00,2015-01-23 15:54:00,2015-01-31 23:57:00,NaN
tpep_dropoff_datetime,32273,2015-01-16 14:18:38.301986,2015-01-01 01:08:00,2015-01-09 07:50:00,2015-01-15 22:57:00,2015-01-23 16:16:00,2015-02-01 13:24:00,NaN
passenger_count,"32,273.00",1.67,0.00,1.00,1.00,2.00,9.00,1.34
trip_distance,"32,273.00",2.77,0.00,1.00,1.65,3.00,56.80,3.36
pickup_longitude,"32,273.00",-72.63,-74.30,-73.99,-73.98,-73.97,0.00,9.88
pickup_latitude,"32,273.00",40.01,0.00,40.74,40.75,40.77,41.02,5.44
RateCodeID,"32,273.00",1.03,1.00,1.00,1.00,1.00,5.00,0.26
dropoff_longitude,"32,273.00",-72.70,-74.39,-73.99,-73.98,-73.96,0.00,9.64
dropoff_latitude,"32,273.00",40.05,0.00,40.73,40.75,40.77,41.17,5.31


## 1 · Filtering & sorting

**Q1 — Trips with more than one passenger, over 1.5 miles, costing more than \$4.**
`SQL: WHERE passenger_count > 1 AND trip_distance > 1.5 AND total_amount > 4`

In [5]:
q1 = trips[(trips['passenger_count'] > 1) &           (trips['trip_distance'] > 1.5) &           (trips['total_amount'] > 4)]len(q1)

5222

**Q2 — Total number of trips.**  `SQL: COUNT(VendorID)`

In [6]:
trips['VendorID'].count()

np.int64(32273)

**Q3 — Trips with more than 3 passengers.**  `SQL: COUNT(VendorID) WHERE passenger_count > 3`

In [7]:
int((trips['passenger_count'] > 3).sum())

3531

**Q4 — Ten longest trips by distance.**  `SQL: ORDER BY trip_distance DESC LIMIT 10`

In [8]:
(trips[['trip_distance', 'tpep_pickup_datetime', 'total_amount']]    .sort_values('trip_distance', ascending=False)    .head(10))

,trip_distance,tpep_pickup_datetime,total_amount
29002,56.80,2015-01-09 14:25:00,270.13
2110,41.94,2015-01-06 11:49:00,205.05
11971,36.54,2015-01-24 00:19:00,180.36
7584,36.07,2015-01-16 03:04:00,145.55
5233,34.30,2015-01-11 03:28:00,15.10
24505,34.20,2015-01-28 19:06:00,141.30
14644,33.60,2015-01-18 07:36:00,58.13
15334,33.52,2015-01-11 05:37:00,138.00
17234,33.30,2015-01-29 11:23:00,63.46
20560,32.90,2015-01-14 03:06:00,100.30


## 2 · Aggregation & grouping

**Q5 — Trip count per payment type.**  `SQL: GROUP BY payment_type`

In [9]:
trips.groupby('payment_type')['payment_type'].count()

,payment_type
payment_type,
1,20057
2,12096
3,101
4,19


**Q6 — Max, min and average fare.**  `SQL: MAX / MIN / AVG(fare_amount)`

In [10]:
trips['fare_amount'].agg(['max', 'min', 'mean'])

,fare_amount
max,234.00
min,-52.00
mean,11.89


**Q7 — Average tip per payment type.**  `SQL: AVG(tip_amount) GROUP BY payment_type`

In [11]:
trips.groupby('payment_type')['tip_amount'].mean()

,tip_amount
payment_type,
1,2.49
2,0.00
3,-0.15
4,0.00


**Q8 — Trip count and total revenue per payment type.**  `SQL: COUNT + SUM(total_amount)`

In [12]:
trips.groupby('payment_type').agg(    trip_count=('VendorID', 'count'),    total_revenue=('total_amount', 'sum'),)

,trip_count,total_revenue
payment_type,,
1,20057,"328,464.96"
2,12096,"148,531.28"
3,101,"1,120.47"
4,19,148.63


## 3 · Time-based analysis

**Q9 — Average trip duration.**  `SQL: AVG(dropoff - pickup)`

In [13]:
trip_duration = trips['tpep_dropoff_datetime'] - trips['tpep_pickup_datetime']trip_duration.mean()

Timedelta('0 days 00:13:06.581352')

**Q10 — Trip count by pickup hour.**  `SQL: GROUP BY EXTRACT(HOUR ...)`

In [14]:
trips.groupby(trips['tpep_pickup_datetime'].dt.hour)['VendorID'].count()

,VendorID
tpep_pickup_datetime,
0,1311
1,718
2,693
3,411
4,196
5,305
6,547
7,1409
8,1341


**Q11 — Average trip distance by day of month.**  `SQL: GROUP BY EXTRACT(DAY ...)`

In [15]:
(trips.groupby(trips['tpep_pickup_datetime'].dt.day)['trip_distance']       .mean()       .sort_values(ascending=False))

,trip_distance
tpep_pickup_datetime,
4,3.51
1,3.50
19,3.33
11,3.11
12,3.09
25,3.08
3,2.95
5,2.95
17,2.89


**Q12 — Average tip by day of month.**  `SQL: AVG(tip_amount) GROUP BY EXTRACT(DAY ...)`

In [16]:
(trips.groupby(trips['tpep_pickup_datetime'].dt.day)['tip_amount']       .mean()       .sort_values(ascending=False))

,tip_amount
tpep_pickup_datetime,
15,1.80
29,1.78
12,1.76
19,1.76
22,1.70
21,1.70
20,1.66
28,1.65
16,1.61


## 4 · Window functions

**Q13 — Trips in each pickup hour, attached to every row.**
`SQL: COUNT(*) OVER (PARTITION BY hour)` → pandas `groupby().transform('count')`

In [17]:
trips['pickup_hour']   = trips['tpep_pickup_datetime'].dt.hourtrips['trips_in_hour'] = trips.groupby('pickup_hour')['pickup_hour'].transform('count')trips[['tpep_pickup_datetime', 'pickup_hour', 'trips_in_hour']].head()

,tpep_pickup_datetime,pickup_hour,trips_in_hour
0,2015-01-15 19:05:00,19,2560
1,2015-01-10 20:33:00,20,1842
2,2015-01-10 20:33:00,20,1842
3,2015-01-10 20:33:00,20,1842
4,2015-01-10 20:33:00,20,1842


**Q14 — Rank payment types by total revenue.**  `SQL: RANK() OVER (ORDER BY SUM(total_amount) DESC)`

In [18]:
payment_totals = trips.groupby('payment_type', as_index=False)['total_amount'].sum()payment_totals['revenue_rank'] = payment_totals['total_amount'].rank(method='min', ascending=False).astype(int)payment_totals.sort_values('revenue_rank')

,payment_type,total_amount,revenue_rank
0,1,"328,464.96",1
1,2,"148,531.28",2
2,3,"1,120.47",3
3,4,148.63,4


**Q15 — Running revenue within each passenger-count group, highest fare first.**
`SQL: SUM(total_amount) OVER (PARTITION BY passenger_count ORDER BY total_amount DESC)`
→ sort, then `groupby().cumsum()`. The final running value per group equals the group total.

In [19]:
running = trips.sort_values(['passenger_count', 'total_amount'], ascending=[True, False]).copy()running['running_revenue'] = running.groupby('passenger_count')['total_amount'].cumsum()running[['passenger_count', 'total_amount', 'running_revenue']].head(10)

,passenger_count,total_amount,running_revenue
18803,0,15.80,15.80
29011,0,15.80,31.60
10458,0,14.15,45.75
20834,0,14.15,59.90
9587,0,12.80,72.70
13174,0,12.30,85.00
19218,0,11.80,96.80
1898,0,11.75,108.55
14487,0,11.15,119.70
9698,0,10.75,130.45


**Q16 — Total revenue per passenger-count group.**  `SQL: SUM(total_amount) OVER (PARTITION BY passenger_count)`

In [20]:
trips['total_per_passenger'] = trips.groupby('passenger_count')['total_amount'].transform('sum')(trips[['passenger_count', 'total_per_passenger']]    .drop_duplicates()    .sort_values('passenger_count'))

,passenger_count,total_per_passenger
1364,0,193.65
0,1,"335,962.33"
9,2,"70,384.08"
7,3,"19,694.94"
48,4,"9,809.45"
16,5,"25,243.20"
29,6,"16,967.09"
5845,9,10.60


## 5 · CTEs & CASE segmentation

**Q17 — Card trips over \$1 that tipped more than \$3.**
`SQL: WITH cte AS (... total_amount>1 AND payment_type=1) SELECT ... WHERE tip_amount>3`

In [21]:
cte = trips[(trips['total_amount'] > 1) & (trips['payment_type'] == 1)]q17 = cte[cte['tip_amount'] > 3]len(q17)

4197

**Q18 — Average tip-to-fare ratio by passenger count (fares > 0).**  `SQL: WITH t AS (tip/fare ...) AVG GROUP BY passenger_count`

In [22]:
tempo = trips[trips['fare_amount'] > 0].copy()tempo['tip_ratio'] = tempo['tip_amount'] / tempo['fare_amount'](tempo.groupby('passenger_count', as_index=False)['tip_ratio']      .mean()      .round(2))

,passenger_count,tip_ratio
0,0,0.17
1,1,0.13
2,2,0.14
3,3,0.12
4,4,0.11
5,5,0.13
6,6,0.13
7,9,0.00


**Q19 — Trip count, average fare and average tip by distance band.**
`SQL: CASE WHEN trip_distance <= 2 ... END`. Bands are `[0, 2], (2, 5], (5, 10], (10, ∞)` —
`pd.cut(include_lowest=True)` keeps 0-mile trips in the first band, matching the SQL cascade exactly.

In [23]:
bands = trips.copy()bands['band'] = pd.cut(    bands['trip_distance'],    bins=[0, 2, 5, 10, float('inf')],    labels=['1: 0-2 mi', '2: 2-5 mi', '3: 5-10 mi', '4: 10+ mi'],    include_lowest=True,)(bands.groupby('band', observed=False)      .agg(trip_count=('VendorID', 'count'),           avg_fare=('fare_amount', 'mean'),           avg_tip=('tip_amount', 'mean'))      .round(2)      .reset_index())

,band,trip_count,avg_fare,avg_tip
0,1: 0-2 mi,19284,7.13,0.93
1,2: 2-5 mi,8889,13.23,1.68
2,3: 5-10 mi,2650,23.81,3.20
3,4: 10+ mi,1450,45.25,5.92


**Q20 — Car-pool trips (2+ passengers) whose pickup longitude is west of its latitude value.**
`SQL: WITH t AS (CASE WHEN passenger_count>1 THEN 'car-pool' ...) WHERE category='car-pool' AND (pickup_longitude - pickup_latitude) < 0`

In [24]:
labelled = trips.copy()labelled['category'] = np.where(labelled['passenger_count'] > 1, 'car-pool', 'individual')q20 = labelled[(labelled['category'] == 'car-pool') &               ((labelled['pickup_longitude'] - labelled['pickup_latitude']) < 0)]len(q20)

9176